In [ ]:
import zipfile
import os
import cv2

zip_path = "/content/archive (21).zip"
extract_path = "extracted_images"

# Extract the ZIP file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# Read all images
images = []

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is not None:
                images.append(img)


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

### Image Preprocessing and Label Extraction

First, we'll preprocess the images by resizing them to a consistent size and normalizing pixel values. We'll also extract the labels based on the directory structure ('yes' or 'no' for tumor presence).

In [ ]:
# Define image size
IMG_SIZE = 128

processed_data = []
labels = []

# Assuming `extract_path` is already defined and contains the extracted images
# from the previous `zipfile` extraction.

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is not None:
                # Resize image
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                # Normalize pixel values to 0-1
                img = img / 255.0
                processed_data.append(img)

                # Extract label from the parent directory name
                # The parent directory name should be 'yes' or 'no'
                label = os.path.basename(root)
                labels.append(label)

# Convert to numpy arrays
X = np.array(processed_data)

# Encode labels: 'no' -> 0, 'yes' -> 1
# We'll use LabelEncoder from sklearn
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(labels)

print(f"Shape of processed data (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}")
print(f"Unique labels: {le.classes_}")
print(f"Encoded labels (first 5): {y[:5]}")

### Feature Extraction using HOG (Histogram of Oriented Gradients)

Instead of raw pixel values, we'll extract HOG features from each image. HOG features are effective for object detection and recognition as they capture edge and texture information. These features will then be used to train our traditional machine learning model.

In [ ]:
hog_features = []
hog_images = []

for img in X:
    # Convert float64 image to uint8 for HOG if necessary, then to grayscale
    # HOG expects grayscale images (single channel)
    # First, convert normalized [0,1] float to [0,255] uint8
    img_uint8 = (img * 255).astype(np.uint8)
    gray_img = cv2.cvtColor(img_uint8, cv2.COLOR_BGR2GRAY)

    # Calculate HOG features and optionally get the HOG image for visualization
    # `block_norm='L2-Hys'` is a common normalization for HOG
    fd, hog_image = hog(gray_img, orientations=9, pixels_per_cell=(8, 8),
                        cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)
    hog_features.append(fd)
    hog_images.append(hog_image)

X_features = np.array(hog_features)

print(f"Shape of HOG features: {X_features.shape}")

# Display a sample HOG image
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(X[0])
plt.title('Original Image (Sample)')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(hog_images[0], cmap='gray')
plt.title('HOG Features Visualization (Sample)')
plt.axis('off')
plt.show()

### Data Splitting and Feature Scaling

Now, we'll split the extracted HOG features and corresponding labels into training and testing sets. We'll also scale the features to ensure that all features contribute equally to the model training, which is crucial for algorithms like SVM.

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

### Training a Support Vector Machine (SVM) Classifier

With the features extracted and scaled, we can now train an SVM model. SVMs are powerful classification algorithms known for their effectiveness in high-dimensional spaces.

In [ ]:
# Initialize and train the SVM classifier
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)

print("SVM model training complete.")

### Evaluate SVM Model Performance

Finally, we'll evaluate the trained SVM model's performance on the test set using various metrics like accuracy, classification report, and a confusion matrix to understand its strengths and weaknesses.

In [ ]:
# Make predictions on the scaled test set
y_pred_svm = svm_model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred_svm)
print(f"Test Accuracy (SVM): {accuracy:.4f}")

print("\nClassification Report (SVM):")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

# Confusion Matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_svm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix (SVM)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()

### Cross-Validation for Robustness Check

To further validate the model's performance and ensure its robustness against potential data leakage or overfitting to a specific train-test split, we will perform **K-Fold Cross-Validation**. This will provide a more generalized estimate of the model's accuracy.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Number of folds for cross-validation
K_FOLDS = 5
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

# Lists to store results from each fold
fold_accuracies = []
fold_reports = []
fold_confusion_matrices = []
misclassified_samples = [] # To store (original_index, true_label, predicted_label)

print(f"Performing {K_FOLDS}-Fold Cross-Validation...")

for fold, (train_index, test_index) in enumerate(skf.split(X_features, y)):
    print(f"\n--- Fold {fold + 1}/{K_FOLDS} ---")

    # Split data for the current fold
    X_train_fold, X_test_fold = X_features[train_index], X_features[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]

    # Scale features for the current fold (fit only on training data)
    scaler_fold = StandardScaler()
    X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
    X_test_fold_scaled = scaler_fold.transform(X_test_fold)

    # Train a new SVM model for the current fold
    svm_model_fold = SVC(kernel='rbf', random_state=42)
    svm_model_fold.fit(X_train_fold_scaled, y_train_fold)

    # Make predictions and evaluate
    y_pred_fold = svm_model_fold.predict(X_test_fold_scaled)
    accuracy_fold = accuracy_score(y_test_fold, y_pred_fold)
    report_fold = classification_report(y_test_fold, y_pred_fold, target_names=le.classes_)
    cm_fold = confusion_matrix(y_test_fold, y_pred_fold)

    print(f"Fold Accuracy: {accuracy_fold:.4f}")
    print("Classification Report:\n", report_fold)

    fold_accuracies.append(accuracy_fold)
    fold_reports.append(report_fold)
    fold_confusion_matrices.append(cm_fold)

    # Identify misclassified samples in this fold
    misclassified_indices = np.where(y_test_fold != y_pred_fold)[0]
    for idx in misclassified_indices:
        original_idx = test_index[idx] # Get the original index from the full dataset
        misclassified_samples.append({
            'original_index': original_idx,
            'true_label': le.inverse_transform([y_test_fold[idx]])[0],
            'predicted_label': le.inverse_transform([y_pred_fold[idx]])[0]
        })

# Summarize cross-validation results
print("\n--- Cross-Validation Summary ---")
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f} +/- {np.std(fold_accuracies):.4f}")
print(f"All Fold Accuracies: {fold_accuracies}")

if misclassified_samples:
    print(f"\nTotal misclassified samples across all folds: {len(misclassified_samples)}")
else:
    print("\nNo misclassified samples found during cross-validation.")

### Misclassified Samples from Cross-Validation

Below are the images that were misclassified by the model during the cross-validation process. Examining these can provide insights into the model's failure cases.

In [ ]:
import matplotlib.pyplot as plt

if misclassified_samples:
    print(f"Displaying {len(misclassified_samples)} misclassified samples:")
    # Calculate number of rows needed for the subplot grid
    num_samples = len(misclassified_samples)
    num_cols = 3 # You can adjust this for desired column count
    num_rows = (num_samples + num_cols - 1) // num_cols

    plt.figure(figsize=(num_cols * 5, num_rows * 5)) # Adjust figure size dynamically

    for i, sample in enumerate(misclassified_samples):
        original_idx = sample['original_index']
        true_label = sample['true_label']
        predicted_label = sample['predicted_label']

        # Get the original image from the preprocessed X array
        img = X[original_idx]

        plt.subplot(num_rows, num_cols, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true_label}\nPred: {predicted_label}", color='red')
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified samples to display. The model achieved 100% accuracy on all folds.")

### Correctly Classified Samples

Here are some examples of images that the model correctly classified. Since the initial model achieved 100% accuracy on its test set, all predictions for these samples will match their true labels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np # Ensure numpy is imported for np.arange
from sklearn.model_selection import train_test_split

# Assuming X, y_test, y_pred_svm are available from the single train-test split evaluation
# We know y_pred_svm was 100% accurate, so all these are correctly classified.

# Select a few samples to display (e.g., first 8 from the test set)
num_samples_to_display = 8

print(f"Displaying {num_samples_to_display} correctly classified samples from the test set:")

# Get the original indices corresponding to the test set from the initial split
# This must be done once, outside the loop.
# We split the array of *indices* (0 to len(X_features)-1) based on X_features and y
# to get the corresponding original indices for the test set.
indices = np.arange(len(X_features))
_, test_indices_initial, _, _ = train_test_split(indices, y, test_size=0.2, random_state=42, stratify=y)

plt.figure(figsize=(15, 8))
plt.clf() # Clear the current figure to ensure a fresh plot

# Iterate up to the number of samples to display, ensuring we don't exceed available test samples
for i in range(min(num_samples_to_display, len(y_test))):
    # original_image_index is the index into the full 'X' dataset
    original_image_index = test_indices_initial[i]

    # img_to_display is the actual image from the 'X' array
    img_to_display = X[original_image_index]

    # Get the true and predicted labels for the current sample from the y_test and y_pred_svm arrays
    # y_test and y_pred_svm are already aligned with the elements of X_test (which corresponds to test_indices_initial)
    true_label_display = le.inverse_transform([y_test[i]])[0]
    predicted_label_display = le.inverse_transform([y_pred_svm[i]])[0]

    plt.subplot(2, 4, i + 1)
    plt.imshow(img_to_display)
    plt.title(f"True: {true_label_display}\nPred: {predicted_label_display}", color='green')
    plt.axis('off')

plt.tight_layout()
plt.show()
print("Correctly classified samples displayed.")

### Predict on an External Image

Now you can upload your own brain tumor image to see how the trained SVM model classifies it. The image will be preprocessed (resized, normalized, HOG features extracted) and then a prediction will be made.

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog

# Upload image
uploaded = files.upload()

# Get the filename of the uploaded image
for fn in uploaded.keys():
  uploaded_filename = fn

# Load and preprocess the uploaded image
print(f"Processing uploaded image: {uploaded_filename}")

# Read the image using OpenCV
# Convert bytes to numpy array
nparr = np.frombuffer(uploaded[uploaded_filename], np.uint8)
external_img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

if external_img is not None:
    # Resize to IMG_SIZE (128x128 as used for training)
    external_img_resized = cv2.resize(external_img, (IMG_SIZE, IMG_SIZE))
    # Normalize pixel values to 0-1
    external_img_normalized = external_img_resized / 255.0

    # Display the uploaded image
    plt.figure(figsize=(6, 6))
    plt.imshow(external_img_normalized)
    plt.title("Uploaded Image")
    plt.axis('off')
    plt.show()

    # Extract HOG features
    img_uint8_hog = (external_img_normalized * 255).astype(np.uint8)
    gray_img_hog = cv2.cvtColor(img_uint8_hog, cv2.COLOR_BGR2GRAY)

    fd_external, _ = hog(gray_img_hog, orientations=9, pixels_per_cell=(8, 8),
                           cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)

    # Scale the features using the previously fitted scaler
    # The scaler was fitted on X_train_scaled
    # We need to reshape fd_external as scaler.transform expects a 2D array
    fd_external_scaled = scaler.transform(fd_external.reshape(1, -1))

    # Make prediction
    prediction_encoded = svm_model.predict(fd_external_scaled)
    predicted_label = le.inverse_transform(prediction_encoded)[0]

    print(f"\nModel Prediction: {predicted_label}")

else:
    print(f"Error: Could not load image from {uploaded_filename}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')